# Allora Forge Builder Kit v2.0 - Topic 69

## 24-Hour Bitcoin Price Prediction

This notebook demonstrates how to:
1. **Use the v2.0 workflow** with Allora data (fast monthly bucket loading)
2. **Train a model** for 1-day Bitcoin price prediction (Topic 69)
3. **Convert log return predictions** to actual price predictions
4. **Package the predict function** for deployment to Allora Network

### Topic 69: 24-Hour Bitcoin Price Prediction
- Predict BTC price 24 hours into the future
- Train on log returns (standard ML practice)
- Deploy predictions as actual prices (what Topic 69 expects)

### What's New in v2.0
- ✅ **Data-source agnostic**: Switch between Binance/Allora easily
- ✅ **Fast loading**: Allora monthly buckets vs slow Binance backfill
- ✅ **Live features**: Automatic 1-min fetching + resampling + feature extraction
- ✅ **Production ready**: Comprehensive testing and documentation


## 📦 Installation

Install the Allora Forge Builder Kit and required dependencies.


In [1]:
%pip install git+https://github.com/allora-network/allora-forge-builder-kit.git
%pip install allora-sdk>=1.0.5 lightgbm scikit-learn pandas numpy matplotlib cloudpickle


  Cloning https://github.com/allora-network/allora-forge-builder-kit.git to /tmp/pip-req-build-ct5me4jn
  Running command git clone --filter=blob:none --quiet https://github.com/allora-network/allora-forge-builder-kit.git /tmp/pip-req-build-ct5me4jn
  Resolved https://github.com/allora-network/allora-forge-builder-kit.git to commit c41e53ab331296c9f9358c826a9fdce33e84d5c8
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for allora-forge-builder-kit: filename=allora_forge_builder_kit-0.1.0-py3-none-any.whl size=14675 sha256=1701278db261c75fa7a15d26a7fda4570fa52c6618a0a319ef5ff59c2de9c57e
  Stored in directory: /tmp/pip-ephem-wheel-cache-exto02g7/wheels/2b/c3/8e/61027f9c7d27b4f0b54de55941fad12ad9b7eda829daecf8b3
Successfully built allora-forge-builder-kit
  Attempting uninstall: allora-forge-builder-kit
    Found existing installation: allora-forge-builder-kit 0.1.0
    Uninstalling allo

## 📚 Imports


In [1]:
from allora_forge_builder_kit import AlloraMLWorkflow
from datetime import datetime, timedelta, timezone
from pathlib import Path
import lightgbm as lgb
import pandas as pd
import numpy as np
import cloudpickle

print("✅ Imports successful!")


✅ Imports successful!


## 🔑 Load Allora API Key

Get your API key from [https://developer.allora.network/](https://developer.allora.network/)


In [2]:
api_key_file = Path(".allora_api_key")
if api_key_file.exists():
    ALLORA_API_KEY = api_key_file.read_text().strip()
    print(f"✅ Loaded Allora API key from {api_key_file}")
else:
    raise FileNotFoundError(f"API key file not found: {api_key_file}")


✅ Loaded Allora API key from .allora_api_key


## ⚙️ Configure Workflow for Topic 69

**Configuration:**
- Asset: Bitcoin (btcusd)
- Lookback: 7 days (168 hours)
- Prediction horizon: 1 day (24 hours)
- Interval: 1-hour bars
- Features: 24 candles × 5 OHLCV = 120 features


In [3]:
# Configuration for 1-day Bitcoin price prediction
TICKER = "btcusd"               # Allora format (lowercase, no separator)
HOURS_NEEDED = 7 * 24           # 7 days lookback
NUMBER_OF_CANDLES = 24          # 24 candles in features
TARGET_LENGTH = 1 * 24          # 1 day (24 hours) prediction horizon
INTERVAL = "1h"                 # 1-hour bars

# Create workflow with Allora data source (v2.0) - faster loading!
workflow = AlloraMLWorkflow(
    tickers=[TICKER],
    hours_needed=HOURS_NEEDED,
    number_of_input_candles=NUMBER_OF_CANDLES,
    target_length=TARGET_LENGTH,
    interval=INTERVAL,
    data_source="allora",       # v2.0: Allora Network (faster than Binance)
    api_key=ALLORA_API_KEY
)

print(f"✅ Workflow configured:")
print(f"   - Asset: {TICKER}")
print(f"   - Lookback: {HOURS_NEEDED} hours ({HOURS_NEEDED//24} days)")
print(f"   - Prediction horizon: {TARGET_LENGTH} hours ({TARGET_LENGTH//24} day)")
print(f"   - Interval: {INTERVAL}")
print(f"   - Features: {NUMBER_OF_CANDLES} candles × 5 OHLCV = {NUMBER_OF_CANDLES*5} features")


✅ Workflow configured:
   - Asset: btcusd
   - Lookback: 168 hours (7 days)
   - Prediction horizon: 24 hours (1 day)
   - Interval: 1h
   - Features: 24 candles × 5 OHLCV = 120 features


## 📥 Backfill Historical Data

Allora uses pre-aggregated **monthly buckets** which load much faster than Binance's 1000-bar API limit.


In [4]:
# Backfill data (Allora uses monthly buckets, very fast!)
start_date = datetime.now(timezone.utc) - timedelta(days=180)  # 6 months
print(f"Fetching data from {start_date.date()} to now...")

workflow.backfill(start=start_date)

print("✅ Backfill complete!")


Fetching data from 2025-04-13 to now...
[workflow] Backfilling ['btcusd'] 2025-04-13 17:23:48.092278+00:00 → now
[Allora backfill-missing] Checking btcusd 2025-04-13 17:23:48.092278+00:00 → 2025-10-10 17:23:48.093068+00:00
[Allora backfill-missing] btcusd: no history at all, full backfill
[Allora backfill] btcusd: 2025-04-13 17:23:48.092278+00:00 → 2025-10-10 17:23:48.093068+00:00
[Allora backfill] btcusd: downloading 6 monthly buckets (filtered from 6)
[Allora backfill] btcusd: fetching API data from 2025-09-30 (10 days, max 15 pages)
[Allora backfill] btcusd: writing 4321 bars to Parquet
[Allora backfill] btcusd: wrote 181 daily parquet files
✅ Backfill complete!


## 🔧 Extract Features and Targets

The workflow extracts:
- **Features**: 120 normalized values (24 candles × 5 OHLCV)
- **Target**: Log returns (24 hours into the future)


In [5]:
# Get full dataset with features and log return targets
df_all = workflow.get_full_feature_target_dataframe_pandas(start_date=start_date)

# Reset index to make it easier to work with
df_all = df_all.reset_index()

print(f"✅ Extracted {len(df_all):,} samples")
print(f"   Columns: {list(df_all.columns[:5])}...")
print(f"   Target: log returns (will be converted to prices for prediction)")


[workflow] Loading data (pandas)
[workflow] Processing btcusd (4320 rows)
✅ Extracted 4,320 samples
   Columns: ['ticker', 'open_time', 'open', 'high', 'low']...
   Target: log returns (will be converted to prices for prediction)


## 📊 Split Train/Validation/Test

Split chronologically: 70% train, 15% validation, 15% test


In [6]:
# 70% train, 15% validation, 15% test
n = len(df_all)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

# Get actual feature columns from DataFrame
feature_cols = [col for col in df_all.columns if col.startswith('feature_')]

X_train = df_all.iloc[:train_end][feature_cols]
y_train = df_all.iloc[:train_end]["target"]

X_val = df_all.iloc[train_end:val_end][feature_cols]
y_val = df_all.iloc[train_end:val_end]["target"]

X_test = df_all.iloc[val_end:][feature_cols]
y_test = df_all.iloc[val_end:]["target"]

print(f"✅ Data split:")
print(f"   Features: {len(feature_cols)} columns")
print(f"   Training:   {len(X_train):>6,} samples ({len(X_train)/n*100:.1f}%)")
print(f"   Validation: {len(X_val):>6,} samples ({len(X_val)/n*100:.1f}%)")
print(f"   Test:       {len(X_test):>6,} samples ({len(X_test)/n*100:.1f}%)")


✅ Data split:
   Features: 120 columns
   Training:    3,024 samples (70.0%)
   Validation:    648 samples (15.0%)
   Test:          648 samples (15.0%)


## 🤖 Train LightGBM Model

Training on log returns (standard ML practice for financial data).


In [7]:
# Train on log returns
model = lgb.LGBMRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=5,
    num_leaves=31,
    random_state=42
)

model.fit(X_train, y_train)

# Evaluate on validation set
val_preds = model.predict(X_val)
val_correlation = np.corrcoef(y_val, val_preds)[0, 1]
val_directional = np.mean(np.sign(y_val) == np.sign(val_preds))

print(f"✅ Model trained!")
print(f"   Validation correlation: {val_correlation:.4f}")
print(f"   Validation directional accuracy: {val_directional:.4f}")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002151 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 30096
[LightGBM] [Info] Number of data points in the train set: 3024, number of used features: 120
[LightGBM] [Info] Start training from score 0.002533
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

## 📈 Evaluate on Test Set


In [8]:
test_preds = model.predict(X_test)
test_correlation = np.corrcoef(y_test, test_preds)[0, 1]
test_directional = np.mean(np.sign(y_test) == np.sign(test_preds))

print(f"✅ Test results:")
print(f"   Correlation: {test_correlation:.4f}")
print(f"   Directional accuracy: {test_directional:.4f}")


✅ Test results:
   Correlation: nan
   Directional accuracy: 0.5170


## 🎯 Create Prediction Function for Topic 69

**Key transformation:** Model predicts log returns, but Topic 69 expects actual prices.

Formula: `predicted_price = current_price * exp(predicted_log_return)`


In [9]:
# Retrain on ALL data for production
model.fit(
    pd.concat([X_train, X_val, X_test]),
    pd.concat([y_train, y_val, y_test])
)
print("✅ Model retrained on full dataset")

# Define prediction function
def predict() -> float:
    """
    Predict Bitcoin price 24 hours into the future (Topic 69).
    
    Returns:
        float: Predicted BTC price in USD
    """
    # Get live features (fetches fresh 1-min data, resamples to 1h, extracts features)
    live_features = workflow.get_live_features(TICKER)
    
    # Predict log return (what the model was trained on)
    predicted_log_return = model.predict(live_features)[0]
    
    # Get current price (last close)
    now = datetime.now(timezone.utc)
    recent_start = now - timedelta(hours=2)  # Get last 2 hours
    raw_data = workflow.load_raw(start=recent_start, end=now)
    current_price = raw_data["close"].iloc[-1]
    
    # Convert log return to predicted price
    # Formula: price_tomorrow = price_today * exp(log_return)
    predicted_price = current_price * np.exp(predicted_log_return)
    
    print(f"\n{'='*60}")
    print(f"Live Prediction for {TICKER}")
    print(f"{'='*60}")
    print(f"Current price:           ${current_price:>12,.2f}")
    print(f"Predicted log return:    {predicted_log_return:>12.6f}")
    print(f"Predicted price (24h):   ${predicted_price:>12,.2f}")
    print(f"Predicted change:        ${predicted_price - current_price:>12,.2f}")
    print(f"Predicted % change:      {(predicted_price/current_price - 1)*100:>12.2f}%")
    print(f"{'='*60}")
    
    return float(predicted_price)

# Test the prediction function
test_prediction = predict()
print(f"\n✅ SUCCESS! Prediction function works.")
print(f"   Predicted BTC price (24h): ${test_prediction:,.2f}")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002564 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 30096
[LightGBM] [Info] Number of data points in the train set: 4320, number of used features: 120
[LightGBM] [Info] Start training from score 0.002000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

## 💾 Save Prediction Function


In [10]:
# Save the prediction function
with open("predict.pkl", "wb") as f:
    cloudpickle.dump(predict, f)
    
print("✅ Prediction function saved to predict.pkl")


✅ Prediction function saved to predict.pkl


## 🚀 Deploy to Allora Network (Topic 69)

Run this cell to start submitting predictions to Topic 69.

When prompted for your mnemonic, hit `<ENTER>` and a wallet will be created automatically in `.allora_key`.


In [11]:
import asyncio
from allora_sdk.worker import AlloraWorker

async def main():
    with open("predict.pkl", "rb") as f:
        predict_fn = cloudpickle.load(f)
    
    worker = AlloraWorker(
        predict_fn=predict_fn,
        api_key=ALLORA_API_KEY,
        topic_id=69  # Topic 69: 24-hour Bitcoin Price Prediction
    )
    
    async for result in worker.run():
        if isinstance(result, Exception):
            print(f"Error: {str(result)}")
        else:
            print(f"Prediction submitted: ${result.prediction:,.2f}")

# Run the worker
await main()


2025-10-10 13:24:20,698 - allora_sdk - INFO - Wallet initialized from LocalWallet
2025-10-10 13:24:22,004 - allora_sdk - INFO - Initialized Allora client for allora-testnet-1
2025-10-10 13:24:22,108 - root - INFO - Worker wallet allo1ueykhylglj6hkz603xz8wqevu3v28xa7ej9485 balance: 0.00 ALLO
2025-10-10 13:24:22,533 - allora_sdk - INFO - Reconnecting...
2025-10-10 13:24:22,920 - allora_sdk - INFO - Websocket connected
